# Marginal Impact v3

Four measures per account, at the group's **1-in-200 OEP**:

| Measure | Question it answers |
|---|---|
| **Last-in marginal** | How much does the group's 1-in-200 drop if we remove this account? |
| **Kernel co-VaR** | How much of the group's 1-in-200 is this account responsible for? |
| **Co-TVaR** | How much of the tail beyond 1-in-200 is this account responsible for? |
| **Standalone** | What is this account's own 1-in-200, ignoring the rest of the group? |

**Terms**

- **Occurrence**: one event happening once in one simulated year, `(SIMID, EVENTID, INSTANCEID)`.
  An event can happen more than once in a year; each instance is a separate occurrence.
- **Year OEP**: the largest single occurrence loss in the year.
- **Rank**: years sorted by OEP, worst first. The 1-in-200 is the year at rank `N_SIMS / 200`.
- **Defining occurrence**: the occurrence that sets a year's OEP.

## 1. Settings

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import betaincinv

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

RETURN_PERIOD = 200

# account ELTs (RMS)
ANLSID    = 112
PERSPCODE = "RL"

# group portfolio ELT (QBE main)
GROUP_TABLE            = "GP_Reporting.MARG.GROUP_ELT_202604_MPML_NAWS"
GROUP_PORT_GROUPING_ID = 354      # GROUP
GROUP_HAZARD_GROUPING  = 352      # North America WS

# reference marginals (QBE main)
REFERENCE_TABLE = "gp_reporting.tmp.MarginalPML_202604"

# simulation (QBE sandbox)
YEQTID       = 69
VENDORVERID  = 33
HAZARDZONEID = 57

# method settings, as fractions of the target rank so they scale with N_SIMS
KERNEL_SPREADS   = [0.10, 0.20]   # 250 and 500 ranks at 500k sims
STANDALONE_BATCH = 25             # accounts simulated together for standalone

OCC = ["SIMID", "EVENTID", "INSTANCEID"]   # one occurrence

## 2. Database connections
Fill in the server and database names.

In [ ]:
from sqlalchemy import create_engine

def mssql_engine(server, database):
    return create_engine(f"mssql+pyodbc://@{server}/{database}"
                         "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes")

engine_rdm = mssql_engine("<RMS_SERVER>",         "<RMS_DATABASE>")      # account ELTs
engine_qbe = mssql_engine("<QBE_MAIN_SERVER>",    "<QBE_MAIN_DATABASE>") # group ELT, reference
engine_ref = mssql_engine("<QBE_SANDBOX_SERVER>", "gp_reference_dev")    # YEQT, VendorEventsMap

## 3. Loading the data
Every loader returns fixed upper-case column names.

In [ ]:
def load_n_sims():
    """
    Number of simulated years, read from the database.
    YEQT has no rows for years without events, so we count distinct years across
    every YEQT in the same simulation set as YEQTID.
    """
    return int(pd.read_sql(f"""
        SELECT COUNT(DISTINCT y.SimID)
        FROM SIM.YEQT y
        JOIN SIM.YEQTInfo yi      ON yi.YEQTID      = y.YEQTID
        JOIN SIM.SimTableInfo sti ON sti.SimTableID = yi.SimTableID
        WHERE sti.NumSimID = (SELECT sti2.NumSimID
                              FROM SIM.YEQTInfo yi2
                              JOIN SIM.SimTableInfo sti2 ON sti2.SimTableID = yi2.SimTableID
                              WHERE yi2.YEQTID = {YEQTID})
    """, engine_ref).iloc[0, 0])


def load_occurrences():
    """Every occurrence in the YEQT, with the vendor event ids the ELTs use."""
    return pd.read_sql(f"""
        SELECT y.SimID AS SIMID, y.EventID AS EVENTID, y.InstanceID AS INSTANCEID,
               y.LossQuantile AS LOSSQUANTILE, v.VENDOREVENTID
        FROM SIM.YEQT y
        JOIN EVT.VENDOREVENTSMAP v ON v.EVENTID = y.EventID
        WHERE y.YEQTID = {YEQTID}
          AND v.VENDORVERID = {VENDORVERID}
          AND v.HAZARDZONEID = {HAZARDZONEID}
    """, engine_ref)


def load_group_elt():
    """Group portfolio ELT: one row per event."""
    return pd.read_sql(f"""
        SELECT VENDOREVENTID, PerspValue AS PERSPVALUE, StdDevI AS STDDEVI,
               StdDevC AS STDDEVC, ExpValue AS EXPVALUE
        FROM {GROUP_TABLE}
        WHERE PortGroupingID = {GROUP_PORT_GROUPING_ID}
          AND HazardZoneGroupingID = {GROUP_HAZARD_GROUPING}
          AND ExpValue > 0
    """, engine_qbe)


def load_account_elts():
    """All account ELTs in one query: one row per (account, event)."""
    return pd.read_sql(f"""
        SELECT ra.ID, ra.EVENTID AS VENDOREVENTID, ra.PERSPVALUE,
               ras.STDDEVI, ras.STDDEVC, ras.EXPVALUE
        FROM DBO.RDM_ACCOUNT ra
        JOIN DBO.RDM_ACCOUNTSTD ras
          ON  ra.ANLSID = ras.ANLSID AND ra.ID = ras.ID
          AND ra.PERSPCODE = ras.PERSPCODE AND ra.EVENTID = ras.EVENTID
          AND ra.PARTITIONID = ras.PARTITIONID
        WHERE ra.ANLSID = {ANLSID} AND ra.PERSPCODE = '{PERSPCODE}' AND ras.EXPVALUE > 0
    """, engine_rdm)


def load_reference():
    """Reference marginals: one row per account group (Accgrpid = account ID)."""
    return pd.read_sql(f"""
        SELECT Accgrpid AS ID, MarginalImpact AS Reference
        FROM {REFERENCE_TABLE}
        WHERE RDM_ANLSID = {ANLSID}
    """, engine_qbe)

In [ ]:
N_SIMS       = load_n_sims()
occurrences  = load_occurrences()
group_elt    = load_group_elt()
account_elts = load_account_elts()
reference    = load_reference()

# a repeated row would silently double losses
assert not group_elt["VENDOREVENTID"].duplicated().any(), "group ELT has repeated events"
assert not account_elts.duplicated(["ID", "VENDOREVENTID"]).any(), "account ELT has repeated rows"

print(f"Simulated years : {N_SIMS:,}")
print(f"Occurrences     : {len(occurrences):,}")
print(f"Group events    : {len(group_elt):,}")
print(f"Accounts        : {account_elts['ID'].nunique():,}")
print(f"Reference rows  : {len(reference):,}")

## 4. ELT to occurrence losses

Each occurrence's loss is read from a beta distribution fitted to the ELT row, at that
occurrence's `LossQuantile`. Group and accounts use the **same** quantile per occurrence.

- `mu = PERSPVALUE / EXPVALUE`, `sigma = (STDDEVI + STDDEVC) / EXPVALUE`
- loss = `EXPVALUE x BetaInverse(LossQuantile; alpha, beta)`

Vendor events that map to the same QBE event are parts of one occurrence, so their losses are added.

In [ ]:
def beta_parameters(mean, stdi, stdc, exposure):
    """Moment-matched beta; variance capped at its maximum if too large (mean preserved)."""
    mu  = np.clip(mean / exposure, 1e-9, 1 - 1e-9)
    var = np.clip(((stdi + stdc) / exposure) ** 2, 1e-12, mu * (1 - mu) * (1 - 1e-6))
    common = mu * (1 - mu) / var - 1
    return mu * common, (1 - mu) * common


def occurrence_losses(elt, occurrences, by=()):
    """Loss per occurrence for every event in `elt`; `by=["ID"]` keeps accounts separate."""
    rows = occurrences.merge(elt, on="VENDOREVENTID")
    alpha, beta = beta_parameters(rows["PERSPVALUE"].to_numpy(), rows["STDDEVI"].to_numpy(),
                                  rows["STDDEVC"].to_numpy(), rows["EXPVALUE"].to_numpy())
    rows["Loss"] = rows["EXPVALUE"].to_numpy() * betaincinv(alpha, beta, rows["LOSSQUANTILE"].to_numpy())
    return rows.groupby(list(by) + OCC, as_index=False)["Loss"].sum()


def nth_largest(values, n):
    """n-th largest value (n = 1 is the largest); 0 if there are fewer than n values."""
    values = np.asarray(values, dtype=float)
    if len(values) < n:
        return 0.0
    return float(np.partition(values, len(values) - n)[len(values) - n])

## 5. Group year OEP and ranks

For every year: its OEP and the occurrence that set it. Years without events are not
listed but still count in `N_SIMS`.

In [ ]:
TARGET_RANK = round(N_SIMS / RETURN_PERIOD)

group_losses = occurrence_losses(group_elt, occurrences)

largest = group_losses.groupby("SIMID")["Loss"].idxmax()
years = (group_losses.loc[largest, OCC + ["Loss"]]
         .rename(columns={"Loss": "OEP"})
         .sort_values("OEP", ascending=False)
         .reset_index(drop=True))
years["Rank"] = np.arange(1, len(years) + 1)

GROUP_200RP = nth_largest(years["OEP"], TARGET_RANK)

print(f"Target rank       : {TARGET_RANK:,}")
print(f"Group 1-in-{RETURN_PERIOD} OEP  : {GROUP_200RP:,.0f}")
print(f"Group AAL         : {group_losses['Loss'].sum() / N_SIMS:,.0f}")

## 6. Account contributions in the worst K years

All accounts' losses for every occurrence in the worst `K` years, scaled so that in each
occurrence they never add up to more than the group's loss:

```
scale        = min(1, group loss / sum of account losses)
contribution = account loss x scale
```

If the accounts add up to less than the group, the gap is "other business".

In [ ]:
def account_contributions(K):
    """Scaled contribution of every account to every occurrence in the worst K years."""
    window_years = years.loc[years["Rank"] <= K, "SIMID"]
    window_occ   = (group_losses[group_losses["SIMID"].isin(window_years)]
                    .rename(columns={"Loss": "GroupLoss"}))

    acct = occurrence_losses(account_elts,
                             occurrences[occurrences["SIMID"].isin(window_years)], by=["ID"])
    acct = acct.merge(window_occ, on=OCC)                 # only occurrences the group has

    scale = np.minimum(1.0, acct["GroupLoss"] / acct.groupby(OCC)["Loss"].transform("sum"))
    acct["Contribution"] = acct["Loss"] * scale
    return acct[["ID"] + OCC + ["Contribution"]], window_occ

## 7. Last-in marginal

Group 1-in-200 **with** the account minus **without** it. For each account: subtract its
contribution from every occurrence it touches, recompute those years' OEP, and re-read rank
`TARGET_RANK` from the worst `K` years.

**Exactness check**: if the new 1-in-200 is still at least the original OEP at rank `K + 1`
(the largest year outside the window), no year from outside could have climbed in, so the
answer is exact. Otherwise `K` doubles and that account is redone.

In [ ]:
def last_in_for_window(contributions, window_occ, K, account_ids):
    base_oep    = years.loc[years["Rank"] <= K].set_index("SIMID")["OEP"]
    outside_max = years["OEP"].iloc[K] if len(years) > K else 0.0     # OEP at rank K+1
    occ_by_year = window_occ.set_index("SIMID")
    by_account  = dict(tuple(contributions.groupby("ID")))

    results = []
    for account_id in account_ids:
        contrib = by_account.get(account_id)
        if contrib is None:
            new_200rp = GROUP_200RP                       # touches none of these years
        else:
            touched = contrib["SIMID"].unique()
            occ = occ_by_year.loc[touched].reset_index().merge(
                contrib[OCC + ["Contribution"]], on=OCC, how="left")
            occ["NetLoss"] = occ["GroupLoss"] - occ["Contribution"].fillna(0.0)

            oep = base_oep.copy()
            oep.loc[touched] = occ.groupby("SIMID")["NetLoss"].max()
            new_200rp = nth_largest(oep.to_numpy(), TARGET_RANK)

        results.append({"ID": account_id, "LastInMarginal": GROUP_200RP - new_200rp,
                        "LastInExact": new_200rp >= outside_max, "LastInK": K})
    return pd.DataFrame(results)


def last_in_marginals(contributions, window_occ, K):
    """All accounts; K doubles only for accounts whose answer is not yet exact."""
    pending, finished = list(account_elts["ID"].unique()), []
    while pending:
        result = last_in_for_window(contributions, window_occ, K, pending)
        finished.append(result[result["LastInExact"]])
        pending = list(result.loc[~result["LastInExact"], "ID"])
        if pending and K >= len(years):                   # whole ranking used: exact
            finished.append(result[~result["LastInExact"]].assign(LastInExact=True))
            break
        if pending:
            K *= 2
            contributions, window_occ = account_contributions(K)
    return pd.concat(finished, ignore_index=True)

## 8. Kernel co-VaR and co-TVaR

Both share out a group figure using each account's contribution in each year's
**defining occurrence** (under OEP, that occurrence is the year's loss).

- **Co-TVaR**: equal weight on the worst `TARGET_RANK` years.
- **Kernel co-VaR**: bell-shaped weight centred on rank `TARGET_RANK`, for two spreads.

Accounts plus "other business" add up to the group figure.

In [ ]:
def rank_weights():
    """Year weights (indexed by SIMID) for each measure; each sums to 1."""
    ranks   = years.set_index("SIMID")["Rank"]
    weights = {"CoTVaR": (ranks <= TARGET_RANK).astype(float)}
    for fraction in KERNEL_SPREADS:
        spread = fraction * TARGET_RANK
        w = np.exp(-0.5 * ((ranks - TARGET_RANK) / spread) ** 2)
        w[(ranks - TARGET_RANK).abs() > 3 * spread] = 0.0
        weights[f"KernelCoVaR_{round(spread)}"] = w
    return {name: (w / w.sum())[w > 0] for name, w in weights.items()}


def allocate(contributions):
    """Each account's share of each group figure, plus the group totals."""
    defining = contributions.merge(years[OCC], on=OCC)          # defining occurrences only
    oep = years.set_index("SIMID")["OEP"]

    shares, totals = [], {}
    for name, w in rank_weights().items():
        share = (defining["SIMID"].map(w).fillna(0.0) * defining["Contribution"]) \
                    .groupby(defining["ID"]).sum().rename(name)
        group_value = float((w * oep.reindex(w.index)).sum())
        shares.append(share)
        totals[name] = {"Group": group_value, "Accounts": share.sum(),
                        "OtherBusiness": group_value - share.sum()}
    return pd.concat(shares, axis=1), pd.DataFrame(totals).T

## 9. Standalone 1-in-200

Each account on its own across all years: its year OEP is its own largest occurrence, read
at rank `TARGET_RANK`. Processed in batches of accounts to keep memory down.

In [ ]:
def standalone_measures():
    ids, rows = account_elts["ID"].unique(), []
    for start in range(0, len(ids), STANDALONE_BATCH):
        batch  = account_elts[account_elts["ID"].isin(ids[start:start + STANDALONE_BATCH])]
        losses = occurrence_losses(batch, occurrences, by=["ID"])
        year_oep = losses.groupby(["ID", "SIMID"])["Loss"].max()
        aal      = losses.groupby("ID")["Loss"].sum() / N_SIMS
        for account_id, oep in year_oep.groupby(level="ID"):
            rows.append({"ID": account_id,
                         "Standalone": nth_largest(oep.to_numpy(), TARGET_RANK),
                         "StandaloneAAL": aal[account_id]})
    return pd.DataFrame(rows)

## 10. Run

In [ ]:
# window: target rank + 3 x the widest kernel spread (4,000 at 500k sims)
K_START = round(TARGET_RANK * (1 + 3 * max(KERNEL_SPREADS)))

contributions, window_occ = account_contributions(K_START)       # computed once, used twice
allocations, group_totals = allocate(contributions)
last_in    = last_in_marginals(contributions, window_occ, K_START)
standalone = standalone_measures()

## 11. Results, with the reference marginals

For each measure: the difference from the reference and the difference as a % of the
reference (blank where the reference is 0).

In [ ]:
MEASURES = ["LastInMarginal"] + [c for c in allocations.columns if c.startswith("KernelCoVaR")] \
           + ["CoTVaR", "Standalone"]

summary = (last_in
           .merge(allocations, left_on="ID", right_index=True, how="left")
           .merge(standalone, on="ID", how="left")
           .fillna({m: 0.0 for m in MEASURES})
           .merge(reference, on="ID", how="outer"))

for m in MEASURES:
    summary[f"{m}_Diff"]    = summary[m] - summary["Reference"]
    summary[f"{m}_DiffPct"] = 100 * summary[f"{m}_Diff"] / summary["Reference"].replace(0, np.nan)

summary = summary.sort_values("Reference", ascending=False).reset_index(drop=True)

print(f"Group 1-in-{RETURN_PERIOD} OEP: {GROUP_200RP:,.0f}  (rank {TARGET_RANK:,} of {N_SIMS:,} years)\n")
display(group_totals)

both = summary.dropna(subset=["Reference", "LastInMarginal"])
print(f"\nAccounts in both: {len(both):,} | only ours: "
      f"{summary['Reference'].isna().sum():,} | only reference: {summary['LastInMarginal'].isna().sum():,}")
display(pd.DataFrame({
    "Mean abs. diff %":   [both[f"{m}_DiffPct"].abs().mean()   for m in MEASURES],
    "Median abs. diff %": [both[f"{m}_DiffPct"].abs().median() for m in MEASURES],
}, index=MEASURES))

display(summary)

## 12. Save

In [ ]:
summary.to_excel(r"C:\Users\245953\Documents\MarginalImpact_202604_v3.xlsx", index=False)
print("Saved")